# White Lodge Savings — mini-warehouse analysis

This notebook answers the two questions that arrived by email, and doubles as
the starting point for ad-hoc questions.

**Before running:** `python -m pipeline.run` (builds the warehouse).

Shortcuts worth memorising before a live session:

| call | what it does |
|---|---|
| `tables()` | everything in the warehouse |
| `columns("marts.fct_claim")` | columns and types of one table |
| `q("select ...")` | SQL → DataFrame |
| `bar` `stacked_bar` `line` `scatter` `heatmap` `kpi` | pre-styled charts |
| `usd(cents)` `pct(fraction)` | formatting |

Money is always **integer cents** in the warehouse. Divide by 100 only when
displaying — `usd()` does that for you.

In [1]:
# Edits to analysis/wls.py take effect on the next cell run, with no kernel
# restart. Without this, `from ... import kpi` binds the function object into
# this namespace and a later edit to the file is invisible — you fix a chart,
# re-run, and see the old one. That costs minutes you don't have in a live
# session.
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

from analysis.wls import (
    q, tables, columns, usd, pct, set_mode,
    bar, stacked_bar, line, scatter, heatmap, kpi,
)

# set_mode("dark")   # the dark palette is selected, not an inversion of light
tables()

,schema,table,rows
0,marts,dim_date,153
1,marts,dim_drug,49
2,marts,dim_partner,8
3,marts,dim_pharmacy,37
4,marts,dq_rejects,2387
5,marts,fct_claim,41400
6,marts,fct_lookup,176721
7,marts,mart_drug_economics,49
8,marts,mart_funnel_daily,2126
9,marts,mart_partner_performance,8


---
## 0. What survived ingestion

Before any business number: how much of the raw data is usable, and what was
blocked. Without that figure in mind, every total below is a guess.

In [2]:
coverage = q("""
    select
        (select count(*) from raw.claims)                                   as raw_rows,
        (select count(*) from marts.fct_claim)                              as analysable,
        (select count(*) from marts.dq_rejects where source_table='claims')  as quarantined
""").iloc[0]

totals = q("""
    select
        count(*)                                    as claims,
        count(*) filter (where is_reverted)          as reverted,
        sum(net_price_cents)                        as gmv,
        sum(net_wls_revenue_cents)                  as wls_revenue,
        count(*) filter (where not has_cost_match)   as no_cost
    from marts.fct_claim
""").iloc[0]

kpi([
    ("analysable claims", f"{coverage.analysable:,}"),
    ("coverage", pct(coverage.analysable / coverage.raw_rows)),
    ("net GMV", usd(totals.gmv)),
    ("White Lodge revenue", usd(totals.wls_revenue)),
    ("reversal rate", pct(totals.reverted / totals.claims)),
], title="The analysable base")

In [3]:
# Why each row was dropped. Every rejection has a named reason and a source file.
q("""
    select source_table, reject_reason, count(*) as rows
    from marts.dq_rejects
    group by 1, 2
    order by 1, 3 desc
""")

,source_table,reject_reason,rows
0,claims,unknown_npi,460
1,claims,duplicate_claim_id,281
2,claims,non_positive_amount,279
3,claims,missing_required_field,148
4,claims,unparseable_timestamp,146
5,claims,unparseable_number,126
6,lookups,unparseable_timestamp,844
7,reverts,orphan_claim_id,53
8,reverts,duplicate_revert_id,26
9,reverts,missing_required_field,13


In [4]:
# Auditable down to the original text: what exactly was "unreadable"?
q("""
    select record_id, raw_payload
    from marts.dq_rejects
    where reject_reason = 'unparseable_number'
    limit 3
""")

,record_id,raw_payload
0,3f73797f-d5bf-4a91-a1e5-101458d74f28,"{""id"":""3f73797f-d5bf-4a91-a1e5-101458d74f28"",""..."
1,0bc462f0-3b98-4205-8e0c-8f12d3cc4e4e,"{""id"":""0bc462f0-3b98-4205-8e0c-8f12d3cc4e4e"",""..."
2,9e0b5a6b-c24c-47f6-a57c-c40df6ac5189,"{""id"":""9e0b5a6b-c24c-47f6-a57c-c40df6ac5189"",""..."


---
## 1. Dale Cooper — partner mix before renegotiations

> *"For a chain of your choice: who's our most valuable partner, and how do they
> compare to the second-best?"*

The question hides a trap: **"most valuable" by which measure?** By claim volume
and by retained revenue the answer is different, because the commercial terms
range from a flat $1.00 cut to 80% of the fee.

Start with the overall picture, then drop into the chain.

In [5]:
partners = q("""
    select
        partner,
        fee_model,
        lookups, claims,
        net_wls_revenue_cents      as wls_revenue,
        net_partner_payout_cents   as payout,
        conversion_rate,
        reversal_rate,
        wls_fee_retention          as retention,
        revenue_cents_per_lookup   as revenue_per_lookup
    from marts.mart_partner_performance
    where claims > 0
    order by wls_revenue desc
""")
partners.assign(
    wls_revenue=lambda d: d.wls_revenue.map(usd),
    payout=lambda d: d.payout.map(usd),
    conversion_rate=lambda d: d.conversion_rate.map(pct),
    reversal_rate=lambda d: d.reversal_rate.map(pct),
    retention=lambda d: d.retention.map(pct),
    revenue_per_lookup=lambda d: d.revenue_per_lookup.map(usd),
)

,partner,fee_model,lookups,claims,wls_revenue,payout,conversion_rate,reversal_rate,retention,revenue_per_lookup
0,Kafka Rx,flat,19694,10534,$63.1k,$10.0k,53.5%,5.3%,86.3%,$3.20
1,Hudi Rx,percentage,56625,11104,$37.6k,$37.7k,19.6%,8.0%,50.0%,$0.66
2,Druid Rx,percentage,58993,5789,$31.2k,$7.8k,9.8%,8.1%,80.0%,$0.53
3,Iceberg Rx,flat,13008,3536,$23.8k,$664.60,27.2%,6.0%,97.3%,$1.83
4,Airflow Rx,flat,11190,1715,$11.2k,$0.00,15.3%,8.6%,100.0%,$1.00
5,Flink Rx,percentage,16325,7899,$11.0k,$44.2k,48.4%,5.0%,20.0%,$0.68
6,direct,none,0,823,$5.7k,$0.00,—,7.3%,100.0%,—


In [6]:
# Part-to-whole: of every dollar of pbm_fee a partner originates, how much stays
# with us and how much walks out. This is the comparison the terms obscure.
stacked_bar(
    partners,
    y="partner",
    series={"wls_revenue": "White Lodge keeps", "payout": "Partner payout"},
    title="Where the pbm_fee goes, by partner",
    note="Net of reversals · full period (Mar–Jul 2026)",
    sort_by="wls_revenue",
)

### Dropping into one chain

The email asks for a specific chain. I pick **`meridian`**, the largest by claim
volume — it's where a renegotiation moves the most money.

In [7]:
chain = "meridian"

by_chain = q(f"""
    select
        partner,
        count(*)                                              as claims,
        sum(net_claim_count)                                  as net_claims,
        sum(net_price_cents)                                  as gmv,
        sum(net_pbm_fee_cents)                                as fee_collected,
        sum(net_partner_fee_cents)                            as payout,
        sum(net_wls_revenue_cents)                            as wls_revenue,
        count(*) filter (where is_reverted) * 1.0 / count(*)   as reversal_rate
    from marts.fct_claim
    where chain = '{chain}'
    group by 1
    order by wls_revenue desc
""")
by_chain.assign(
    gmv=lambda d: d.gmv.map(usd),
    fee_collected=lambda d: d.fee_collected.map(usd),
    payout=lambda d: d.payout.map(usd),
    wls_revenue=lambda d: d.wls_revenue.map(usd),
    reversal_rate=lambda d: d.reversal_rate.map(pct),
)

,partner,claims,net_claims,gmv,fee_collected,payout,wls_revenue,reversal_rate
0,Kafka Rx,3253,3052.0,$16.6M,$22.5k,$3.1k,$19.5k,6.2%
1,Hudi Rx,3046,2759.0,$11.6M,$20.2k,$10.1k,$10.1k,9.4%
2,Druid Rx,1524,1364.0,$7.4M,$10.0k,$2.0k,$8.0k,10.5%
3,Iceberg Rx,904,849.0,$4.6M,$6.2k,$169.80,$6.0k,6.1%
4,Airflow Rx,412,372.0,$1.2M,$2.6k,$0.00,$2.6k,9.7%
5,Flink Rx,1764,1659.0,$8.9M,$12.3k,$9.9k,$2.5k,6.0%
6,direct,236,216.0,$956.6k,$1.5k,$0.00,$1.5k,8.5%


In [8]:
bar(
    by_chain,
    x="wls_revenue", y="partner",
    title=f"White Lodge net revenue in the {chain} chain, by partner",
    note="After the partner payout and net of reversals",
    xtitle="retained revenue",
)

In [9]:
# The full chain × partner picture, so the chain isn't chosen blind.
grid = q("""
    select chain, partner, sum(net_wls_revenue_cents) as revenue
    from marts.fct_claim
    group by 1, 2
""")
heatmap(grid, x="partner", y="chain", z="revenue",
        title="Revenue retained by White Lodge — chain × partner",
        note="Darker is more revenue · blank cells had no claims")

---
## 2. Gordon Cole — where the margin would come from

> *"If we wanted to increase White Lodge's margin, what would you suggest —
> based on what you're seeing in the data?"*

The answer is in one thing that shows up the moment you put price and fee on the
same chart.

In [10]:
sample = q("""
    select price_cents / 100.0 as price, pbm_fee_cents / 100.0 as fee
    from marts.fct_claim
    where not is_reverted
""")

scatter(
    sample, x="price", y="fee",
    title="What we charge has no relationship to the value we intermediate",
    note="One point per claim · price axis on a log scale",
    xtitle="claim price (log)", ytitle="pbm_fee charged",
    log_x=True, money_x=True, money_y=True,
)

In [11]:
# The same fact, quantified: take rate collapses as the claim grows.
bands = q("""
    select
        case
            when price_cents < 10000    then '1 · up to $100'
            when price_cents < 1000000  then '2 · $100 to $10k'
            else                             '3 · above $10k'
        end                                         as band,
        count(*)                                    as claims,
        sum(price_cents)                            as gmv,
        sum(pbm_fee_cents)                          as fee,
        sum(pbm_fee_cents) * 1.0 / sum(price_cents)  as take_rate
    from marts.fct_claim
    where not is_reverted
    group by 1
    order by 1
""")
bands.assign(gmv=lambda d: d.gmv.map(usd), fee=lambda d: d.fee.map(usd),
             take_rate=lambda d: d.take_rate.map(lambda v: f"{v*100:.4f}%"))

,band,claims,gmv,fee,take_rate
0,1 · up to $100,24143,$561.5k,$180.5k,32.1503%
1,2 · $100 to $10k,12757,$14.5M,$86.9k,0.5982%
2,3 · above $10k,1761,$185.6M,$16.5k,0.0089%


In [12]:
# Two measures on incomparable scales (GMV in millions, take rate in thousandths
# of a percent) => two charts. Never a secondary axis.
bar(bands, x="gmv", y="band",
    title="Where the financial volume is",
    note="Net GMV by claim value band", xtitle="GMV")

In [13]:
bar(bands.assign(take_bp=lambda d: d.take_rate * 10000), x="take_bp", y="band",
    money=False,
    title="Where our compensation is",
    note="Take rate in basis points (1 bp = 0.01%) — same ordering as the chart above",
    xtitle="take rate (basis points)")

### The second lever: generic substitution

For every brand drug, NADAC publishes the cost of the equivalent generic. That
lets us estimate how much acquisition cost would leave the chain if the same
fill were dispensed as the generic.

In [14]:
generics = q("""
    select
        ndc, ndc_description, drug_class,
        net_claims                          as claims,
        net_gmv_cents                       as gmv,
        net_wls_revenue_cents               as wls_revenue,
        generic_substitution_savings_cents  as potential_saving,
        reversal_rate
    from marts.mart_drug_economics
    where generic_substitution_savings_cents > 0
    order by potential_saving desc
    limit 10
""")
generics.assign(gmv=lambda d: d.gmv.map(usd), wls_revenue=lambda d: d.wls_revenue.map(usd),
                potential_saving=lambda d: d.potential_saving.map(usd),
                reversal_rate=lambda d: d.reversal_rate.map(pct))

,ndc,ndc_description,drug_class,claims,gmv,wls_revenue,potential_saving,reversal_rate
0,61958220101,EPCLUSA 400 MG-100 MG TABLET,brand,602.0,$39.6M,$3.3k,$24.3M,7.7%
1,10631011831,ABSORICA 40 MG CAPSULE,brand,1100.0,$3.2M,$7.0k,$2.4M,7.2%
2,00078050161,EXELON 4.6 MG/24HR PATCH,brand,1622.0,$2.6M,$11.6k,$2.2M,7.6%
3,50419045304,CLIMARA 0.075 MG/DAY PATCH,brand,1072.0,$1.5M,$874.23,$413.1k,5.9%
4,00002418430,EVISTA 60 MG TABLET,brand,550.0,$262.6k,$1.6k,$225.9k,7.1%
5,70165001530,ADZENYS XR-ODT 9.4 MG TABLET,brand,572.0,$850.8k,$4.2k,$115.6k,5.5%


In [15]:
bar(generics.head(6), x="potential_saving", y="ndc_description",
    title="Acquisition cost that generic substitution would remove",
    note="Gap between brand NADAC and the equivalent generic, at dispensed volume",
    xtitle="estimated saving")

### The third: reversals

A reversal is revenue that was earned and handed back. Unlike margin that never
existed, this one is recoverable.

In [16]:
reversals = q("""
    select
        sum(wls_net_fee_cents) filter (where is_reverted) as revenue_lost,
        sum(wls_net_fee_cents)                            as revenue_potential,
        median(hours_to_revert)                           as hours_to_revert
    from marts.fct_claim
""").iloc[0]

kpi([
    ("revenue lost to reversals", usd(reversals.revenue_lost)),
    ("% of potential revenue", pct(reversals.revenue_lost / reversals.revenue_potential)),
    ("median time to reversal", f"{reversals.hours_to_revert / 24:.1f} days"),
], title="What reversals cost")

In [17]:
weekly = q("""
    select week_start, partner,
           sum(claims) as claims, sum(reverted_claims) as reverted
    from marts.mart_funnel_daily
    where partner <> 'unknown'
    group by 1, 2
""")
weekly["rate"] = weekly.reverted / weekly.claims

line(weekly, x="week_start", y="rate", color="partner",
     title="Reversal rate by partner, by week",
     note="Flat across all of them — a structural cost, not a one-off incident",
     ytitle="reversal rate")

---
## Scratch area

Room for whatever arrives live. The shortest path is almost always
`marts.fct_claim` on its own — it already carries `chain`, `partner`, `channel`
and `drug_class` denormalised for exactly this.

In [18]:
columns("marts.fct_claim")

,column,type
0,claim_id,VARCHAR
1,pharmacy_npi,VARCHAR
2,chain,VARCHAR
3,ndc,VARCHAR
4,partner,VARCHAR
5,channel,VARCHAR
6,drug_class,VARCHAR
7,filled_at,TIMESTAMP
8,filled_date,DATE
9,quantity,DOUBLE


In [19]:
q("""
    select *
    from marts.fct_claim
    limit 5
""")

,claim_id,pharmacy_npi,chain,ndc,partner,channel,drug_class,filled_at,filled_date,quantity,...,looked_up_at,minutes_lookup_to_fill,_source_file,pharmacy_margin_cents,generic_substitution_savings_cents,net_price_cents,net_pbm_fee_cents,net_partner_fee_cents,net_wls_revenue_cents,net_claim_count
0,d324ddeb-6ee3-4a55-b47a-6ae507f96a9d,4984991735,beacon,42806002001,Hudi Rx,website,generic,2026-03-01 12:17:54,2026-03-01,100.0,...,2026-03-01 11:57:58,20,C:\Users\luisc\Documents\white_lodge_savings\s...,158,<NA>,1405,959,480,479,1
1,69ecdd2f-a880-4665-9122-e0d673dc5852,7260900195,sunrise,42806002001,Flink Rx,integration,generic,2026-03-01 01:26:38,2026-03-01,90.0,...,2026-03-01 00:37:19,49,C:\Users\luisc\Documents\white_lodge_savings\s...,130,<NA>,1238,848,678,170,1
2,f30a73a9-bfa1-42e9-9e53-017be74eb0ec,7694029884,union,42806002001,Airflow Rx,website,generic,2026-03-01 04:46:55,2026-03-01,45.0,...,2026-03-01 03:50:14,56,C:\Users\luisc\Documents\white_lodge_savings\s...,43,<NA>,1002,829,0,829,1
3,5cf16365-5f71-40d2-b98f-fcb4c45fd1a8,9582170077,meridian,42806002001,Flink Rx,integration,generic,2026-03-01 11:36:32,2026-03-01,60.0,...,2026-03-01 11:07:05,29,C:\Users\luisc\Documents\white_lodge_savings\s...,167,<NA>,1309,969,775,194,1
4,d5bc9060-9e7a-465b-8531-616b0012a536,9854497121,beacon,42806002001,Flink Rx,integration,generic,2026-03-02 13:39:46,2026-03-02,30.0,...,2026-03-02 13:36:23,3,C:\Users\luisc\Documents\white_lodge_savings\s...,67,<NA>,1132,978,782,196,1
